In [ ]:
import ee
import geemap
import geopandas as gpd
import pandas as pd


# Step 0: Initialize GEE

ee.Authenticate()
ee.Initialize(project='heat-wave-483115')


# Step 1: Loading ROI SOUSSE

delegations = gpd.read_file("/content/sousse.geojson").to_crs(epsg=4326)
governorate_geom = delegations.geometry.union_all()

if governorate_geom.is_empty:
    raise ValueError("Combined governorate geometry is empty. Check your 'sousse.geojson' file.")


governorate_proj = gpd.GeoSeries([governorate_geom], crs="EPSG:4326").to_crs(epsg=32632)
governorate_simplified = governorate_proj.simplify(30).to_crs(epsg=4326).iloc[0]

geojson_data = governorate_simplified.__geo_interface__
if geojson_data['type'] == 'Polygon':
    geojson_data['coordinates'] = [[[c[0], c[1]] for c in ring] for ring in geojson_data['coordinates']]
elif geojson_data['type'] == 'MultiPolygon':
    geojson_data['coordinates'] = [[[[c[0], c[1]] for c in ring] for ring in poly] for poly in geojson_data['coordinates']]

aoi = ee.Geometry(geojson_data)
print("AOI ready.")

AOI ready.
Export started. Check status with export_task.status()


In [ ]:
# Step 2: Building feature image (Sentinel-2 bands + spectral indices)

s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi)
    .filterDate("2023-01-01", "2023-12-31")
    .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
    .median()
    .clip(aoi)
)

ndvi = s2.normalizedDifference(["B8", "B4"]).rename("NDVI")
ndbi = s2.normalizedDifference(["B11", "B8"]).rename("NDBI")
ndwi = s2.normalizedDifference(["B3", "B8"]).rename("NDWI")
features_img = s2.select(["B2", "B3", "B4", "B8", "B11", "B12"]).addBands([ndvi, ndbi, ndwi])

BANDS = ["B2", "B3", "B4", "B8", "B11", "B12", "NDVI", "NDBI", "NDWI"]


# Step 3: Training data — Source 1: ESA WorldCover (whole governorate, both classes)

worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().clip(aoi)
label_img = worldcover.select("Map").eq(50).rename("built_up")
training_stack = features_img.addBands(label_img)

esa_samples = training_stack.stratifiedSample(
    numPoints=1000,
    classBand="built_up",
    region=aoi,
    scale=10,
    seed=42,
    geometries=True,
)

In [ ]:
# Step 4: Training data — Source 2: my own infrastructure evidence
# Lamps + streets = confirmed built_up=1 (real ground truth).


lamp_gdf = gpd.read_file("/content/sousse_lamps.geojson").to_crs(epsg=4326)
streets_gdf = gpd.read_file("/content/sousse_streets.geojson").to_crs(epsg=4326)

lamp_buffered = lamp_gdf.to_crs(epsg=32632).buffer(15).to_crs(epsg=4326)
streets_buffered = streets_gdf.to_crs(epsg=32632).buffer(10).to_crs(epsg=4326)

own_evidence_geom = gpd.GeoSeries(
    pd.concat([lamp_buffered, streets_buffered], ignore_index=True)
).union_all()


own_evidence_simplified = gpd.GeoSeries([own_evidence_geom], crs="EPSG:4326") \
    .to_crs(epsg=32632).simplify(5).to_crs(epsg=4326).iloc[0]

own_evidence_ee = ee.Geometry(own_evidence_simplified.__geo_interface__)

own_samples_raw = features_img.addBands(ee.Image.constant(1).rename("built_up")).sample(
    region=own_evidence_ee,
    scale=10,
    numPixels=1000,
    seed=42,
    geometries=True,
)

In [ ]:
# Step 5: Combinig training data + trainig Random Forest

training_samples = esa_samples.merge(own_samples_raw)

classifier = ee.Classifier.smileRandomForest(numberOfTrees=300).train(
    features=training_samples,
    classProperty="built_up",
    inputProperties=BANDS,
)

# Step 5b: Spliting EACH source separately (so both are represented in test)

esa_samples = esa_samples.randomColumn("random", seed=1)
esa_train = esa_samples.filter(ee.Filter.lt("random", 0.7))
esa_test = esa_samples.filter(ee.Filter.gte("random", 0.7))

own_samples_raw = own_samples_raw.randomColumn("random", seed=2)
own_train = own_samples_raw.filter(ee.Filter.lt("random", 0.7))
own_test = own_samples_raw.filter(ee.Filter.gte("random", 0.7))


# Step 5c: Merging into ONE combined train set and ONE combined test set

train_set = esa_train.merge(own_train)
test_set = esa_test.merge(own_test)   # <-- this is the "overall" test set

print("Train size:", train_set.size().getInfo())
print("Test size (combined, both sources):", test_set.size().getInfo())


# Step 5d: Training on combined train_set

classifier = ee.Classifier.smileRandomForest(numberOfTrees=300).train(
    features=train_set,
    classProperty="built_up",
    inputProperties=BANDS,
)


# Step 5e: Classifying the COMBINED test_set and compute OVERALL accuracy

test_classified = test_set.classify(classifier)

confusion_matrix = test_classified.errorMatrix("built_up", "classification")

print("Confusion Matrix (rows=actual, cols=predicted):")
print(confusion_matrix.getInfo())

print("Overall Accuracy (ESA + own samples combined):", confusion_matrix.accuracy().getInfo())
print("Kappa Coefficient:", confusion_matrix.kappa().getInfo())
print("Producer's Accuracy per class:", confusion_matrix.producersAccuracy().getInfo())
print("Consumer's Accuracy per class:", confusion_matrix.consumersAccuracy().getInfo())

# Step 6: Applying the trained model to EVERY pixel in the governorate

classified = features_img.select(BANDS).classify(classifier).rename("built_up")


In [ ]:
# Step 7: Exporting raster mask to Google Drive

export_task = ee.batch.Export.image.toDrive(
    image=classified,
    description='sousse_built_up_prediction',
    folder='EarthEngine_exports',
    fileNamePrefix='sousse_built_up_prediction',
    region=aoi,
    scale=10,
    maxPixels=1e10,
    fileFormat='GeoTIFF',
)
export_task.start()
print("Export started. Check status with export_task.status()")

In [ ]:
#checking raster exporting status
import time
while export_task.active():
    print("Still running... status:", export_task.status()['state'])
    time.sleep(20)
print("Finished:", export_task.status())

Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still running... status: RUNNING
Still runn

In [ ]:
#Step 8: Converting the built up tif into polygons

import rasterio
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape

with rasterio.open("/content/sousse_built_up_prediction (1).tif") as src:
    image = src.read(1)  # band 1
    transform = src.transform
    crs = src.crs

    results = (
        {"properties": {"built_up": int(v)}, "geometry": s}
        for s, v in shapes(image, mask=(image == 1), transform=transform)
    )

    geoms = list(results)

gdf = gpd.GeoDataFrame.from_features(geoms, crs=crs)
gdf.to_file("sousse_built_up.shp")
print(f"Saved {len(gdf)} polygons to sousse_built_up.shp")

Saved 235970 polygons to sousse_built_up.shp
